In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/img-capstone/AI_gen_img_capstone.png



#### Problem : Doctor has to devote his time in writing prescription for the patient. Even the Doctor writes in hurry , where the prescription is not even readable (patient cannot get what is written on the precription).?

#### Solution : Agent which writes an accurate authentic prescription for Doctor and waits for approval by Doctor and lastly prints it.This saves Doctor's time and let's it devote more time to other patients , where patients even can read the written prescription.


Workflow:

Agent 1 : Ayurvedic prescriber 

Agent 2 : Homeopathic prescriber 

Agent 3 : Allopathic prescriber 

these agents work in parallel and give indivisual outputs to Aggregator Agent which in turn summarizes it and hands over to critique agent which are Looped in a Refinement loop where the prescription is finalized and presented to Doctor waiting for HILP response.
After confirmation by Doctor , prescription is printed , ready to be given to patient.


![Workflow of Agents](http://https://www.kaggle.com/datasets/shivapipy/img-capstone)

## ⚙️ Section 1: Setup
### **Install dependencies**
The Kaggle Notebooks environment includes a pre-installed version of the google-adk library for Python and its required dependencies, so we don't need to install additional packages in this notebook.

In [2]:
#!pip install google-adk

###  1.1 Configure your Gemini API Key.
1.This notebook uses the Gemini API, which requires authentication.

2.Authenticate in the notebook.

Run the cell below to complete authentication.

In [3]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

✅ Gemini API key setup complete.


### 1.2 Import ADK components

Now, import the specific components we will need from the Agent Development Kit and the Generative AI library. This keeps our code organized and ensures we have access to the necessary building blocks.

In [4]:
import uuid
from google.genai import types

from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.3 Helper functions

We'll define some helper functions. If you are running this outside the Kaggle environment, you don't need to do this.

In [5]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]['base_url']

    try:
        path_parts = baseURL.split('/')
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix

print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.4: Configure Retry Options# 

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [6]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

### Our Multi-Agent

**The Problem: The "Do-It-All" Agent**

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to write prescription for all [yypes of medicinal paradigms] and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

**The Solution: A Team of Specialists**

Instead of one "do-it-all" agent, we can build a multi-agent system. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent only does write prescription for homeopathy, another only for ayurved). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

**Architecture: Single Agent vs Multi-Agent Team**

## 2.1 Prescription Writer System
Let's build a system with three specialized agents:

**Homeopathic Agent**- Suggest Homeopathic Medicines for the Disease.

**Ayurvedic Agent** - Suggest Ayurvedic Medicines for the Disease.

**Allopathic Agent** - Suggest Allopathic Medicines for the Disease.

**Refiner Agent** - Checks the Authenticity , accuracy of the prescription.

**HILP** - Doctor approves the prescrption to be printed.

## 2.2 Define agents
Now, let's build our agent. We'll configure an Agent by setting its key properties, which tell it what to do and how to operate.

To learn more, check out the documentation related to agents in ADK.

These are the main properties we'll set:


* **name** and **description**: A simple name and description to identify our agent.
* **model**: The specific LLM that will power the agent's reasoning. We'll use "gemini-2.5-flash-lite".
* **instruction**: The agent's guiding prompt. This tells the agent its goal is and how to behave.
* **tools**: A list of tools that the agent can use. To start, we'll give it the google_search tool, which lets it find up-to-date information online.
  

## 2.3 : Parallel Workflows - Independent Researchers

We have several tasks that are not dependent on each other.

The Solution: Concurrent Execution

When you have independent tasks, you can run them all at the same time using a ParallelAgent. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

Use Parallel when: Tasks are independent, speed matters, and you can execute concurrently

In [7]:
# Homeopathic Agent: Its job is to use the google_search tool and present medicines.
homeopathic_agent = Agent(
    name="HomeopathicAgent",
    model = Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request medicines with proper dosage.
    instruction="""You are a specialized Homeopathic Doctor. Your only job is to use the
    google_search tool to find 2-3 relevant Homeopathic medicines with proper dosage for the given disease and present the medicines and dosage.""",
    tools=[google_search],
    output_key="homeopathic_findings", # The result of this agent will be stored in the session state with this key.
)

print("✅ homeopathic_agent created.")

✅ homeopathic_agent created.


In [8]:
# Ayurvedic Agent: Its job is to use the google_search tool and present medicines.
ayurvedic_agent = Agent(
    name="AyurvedicAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request medicines with proper dosage.
    instruction="""You are a specialized Ayurvedic Doctor. Your only job is to use the
    google_search tool to find 2-3 relevant Ayurvedic medicines with proper dosage for the given disease and present the medicines and dosage.""",
    tools=[google_search],
    output_key="ayurvedic_findings",# The result of this agent will be stored in the session state with this key.
)

print("✅ ayurvedic_agent created.")

✅ ayurvedic_agent created.


In [9]:
# Allopathic Agent: Its job is to use the google_search tool and present medicines..
allopathic_agent = Agent(
    name="AllopathicAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request medicines with proper dosage.
    instruction="""You are a specialized Allopathic Doctor. Your only job is to use the
    google_search tool to find 2-3 relevant Allopathic medicines with proper dosage for the given disease and present the medicines and dosage.""",
    tools=[google_search],
    output_key="allopathic_findings",# The result of this agent will be stored in the session state with this key.
)

print("✅ allopathic_agent created.")

✅ allopathic_agent created.


👉 Then we bring the agents together under a parallel agent, which is itself nested inside of a sequential agent.

This design ensures that the research agents run first in parallel, then once all of their research is complete, the aggregator agent brings together all of the research findings into a single report:

In [10]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""Combine these three medical prescriptions into a single executive prescription:

    **Homeopathic prescription:**
    {homeopathic_findings}
    
    **Ayurvedic prescription:**
    {ayurvedic_findings}
    
    **Allopathic prescription:**
    {allopathic_findings}
    
    Your prescription should contain only correct medicines with dosages for disease , and the tagline 'This is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes.'. The final prescription should be around 20 to 30 words.""",
    output_key="prescription_summary", # This will be the final output of the entire system.
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


In [11]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[ayurvedic_agent,homeopathic_agent,allopathic_agent],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
shoot_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


## 2.4: Loop Workflows - The Refinement Cycle¶
ParallelAgent produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our prescription is bad? We have no way to review it and ask for a rewrite.

The Solution: Iterative Refinement

When a task needs to be improved through cycles of feedback and revision, you can use a LoopAgent. A LoopAgent runs a set of sub-agents repeatedly until a specific condition is met or a maximum number of iterations is reached. This creates a refinement cycle, allowing the agent system to improve its own work over and over.

Use Loop when: Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

In [12]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a Medical practitioner and Doctor critic. Review the medical prescription provided below.
    Prescription: {prescription_summary}

    Evaluate the prescription's medicines and it's dosage. 
    -The medicines prescribed should be highly accurate.
    -The prescription should have dosage for each medicine.
    -Citations to be included in prescription.
    If the prescription is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique", # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


Now, we need a way for the loop to actually stop based on the critic's feedback. The LoopAgent itself doesn't automatically know that "APPROVED" means "stop."

We need an agent to give it an explicit signal to terminate the loop.

We do this in two parts:


* A simple Python function that the LoopAgent understands as an "exit" signal.
* An agent that can call that function when the right condition is met.

First, we'll define the exit_loop function:

In [13]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the prescription is finished and no more changes are needed."""
    return {"status": "approved", "message": "Prescription approved. Exiting refinement loop."}

print("✅ exit_loop function created.")

✅ exit_loop function created.


To let an agent call this Python function, we wrap it in a FunctionTool. Then, we create a RefinerAgent that has this tool.

👉 Notice its instructions: this agent is the "brain" of the loop. It reads the {critique} from the CriticAgent and decides whether to (1) call the exit_loop tool or (2) rewrite the prescription.

In [14]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a medical prescrition refiner. You have a prescription draft and critique.
    
    Prescription Draft: {prescription_summary}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the prescription draft to fully incorporate the feedback from the critique.""",
    
    output_key="prescription_summary", # It overwrites the story with the new, refined version.
    tools=[FunctionTool(exit_loop)], # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


Then we bring the agents together under a loop agent, which is itself nested inside of a sequential agent.

This design ensures that the system first produces an initial story draft, then the refinement loop runs up to the specified number of max_iterations:

In [15]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
prescription_refinement_loop = LoopAgent(
    name="PrescriptionRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2, # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: AggregatorAgent -> Refinement Loop.
froot_agent = SequentialAgent(
    name="PrescriptionPipeline",
    sub_agents=[shoot_agent , prescription_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


In [16]:
def place_prescription_order(
   number : int, quality : str, tool_context: ToolContext
) -> dict:
    """Places a prescription order. Requires approval by a Doctor.

    Args:
        quality : Whether approved by Doctor or not

    Returns:
        Dictionary with prescription status
    """

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # SCENARIO : This is the time this tool is called. Large orders need human approval - PAUSE here.
    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"⚠️ Prescrition : {number} . Do you want to approve?",
        )
        return {  # This is sent to the Agent
            "status": "pending",
            "message": f"This prescription requires approval",
        }

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
 # -----------------------------------------------------------------------------------------------
    # SCENARIO 3: The tool is called AGAIN and is now resuming. Handle approval response - RESUME here.
    if tool_context.tool_confirmation.confirmed:
        return {
            "status": "approved",
            "order_id": f"ORD-{number}-HUMAN",
            "quality": quality ,
            "message": f"Prescription approved",
        }
    else:
        return {
            "status": "rejected",
            "message": f"Prescription rejected",
        }


print("✅ Long-running functions created!")

✅ Long-running functions created!


How the Three Scenarios Work
The tool handles two scenarios by checking tool_context.tool_confirmation:

Scenario 1: Prescription order - FIRST CALL

Tool detects it's a first call: if not tool_context.tool_confirmation:
Calls request_confirmation() to request human approval
Returns {'status': 'pending', ...} immediately
ADK automatically creates adk_request_confirmation event
Agent execution pauses - waiting for human decision
Scenario 2: Prescription check - RESUMED CALL

Tool detects it's resuming: if not tool_context.tool_confirmation: is now False
Checks human decision: tool_context.tool_confirmation.confirmed
If True → Returns approved status
If False → Returns rejected status
Key insight: Between the two calls, your workflow code must detect the adk_request_confirmation event and resume with the approval decision

In [17]:
# Create shipping agent with pausable tool
confirming_agent = LlmAgent(
    name="confirming_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a Compounder , Doctors' coordinator assistant.
  
  When Doctor request to prescription:
   1. Use the place_prescription_order tool with the status of prescription 
   2. If the order status is 'pending', inform the user that approval is required
   3. After receiving the final result, provide a clear summary including:
      - Order status (approved/rejected)
      - Order ID (if available)
   4. Keep responses concise but informative
  """,
    tools=[FunctionTool(func=place_prescription_order)],
)

print("✅ Confirming Agent created!")

✅ Confirming Agent created!


Step 2: Wrap in resumable App

The problem: A regular LlmAgent is stateless - each call is independent with no memory of previous interactions. If a tool requests approval, the agent can't remember what it was doing.

The solution: Wrap your agent in an App with resumability enabled. The App adds a persistence layer that saves and restores state.

What gets saved when a tool pauses:

All conversation messages so far
Which tool was called (place_prescription_order)
Tool parameters (5 , NotApproved)
Where exactly it paused (waiting for approval)
When you resume, the App loads this saved state so the agent continues exactly where it left off - as if no time passed.

In [18]:
# Wrap the agent in a resumable app - THIS IS THE KEY FOR LONG-RUNNING OPERATIONS!
prescription_app = App(
    name="prescription_coordinator",
    root_agent=confirming_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

print("✅ Resumable app created!")

✅ Resumable app created!


/tmp/ipykernel_13/1223343667.py:5: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=True),


Step 3: Create Session and Runner with the App

Pass app=shipping_app instead of agent=... so the runner knows about resumability.

In [19]:
session_service = InMemorySessionService()

# Create runner with the resumable app
prescription_runner = Runner(
    app=prescription_app,  # Pass the app instead of the agent
    session_service=session_service,
)

print("✅ Runner created!")

✅ Runner created!


## 2.5 : Building the Workflow¶
‼️ Important: The workflow code uses ADK concepts like Sessions, Runners, and Events. We'll cover what you need to know for long-running operations in this notebook.

⚠️ The Critical Part - Handling Events in Your Workflow
The agent won't automatically handle pause/resume. Every long-running operation workflow requires you to:

Detect the pause: Check if events contain adk_request_confirmation
Get human decision: In production, show UI and wait for user click. Here, we simulate it.
Resume the agent: Send the decision back with the saved invocation_id

Understand Key Technical Concepts
👉 events - ADK creates events as the agent executes. Tool calls, model responses, function results - all become events

👉 adk_request_confirmation event - This event is special - it signals "pause here!"

Automatically created by ADK when your tool calls request_confirmation()
Contains the invocation_id
Your workflow must detect this event to know the agent paused
👉 invocation_id - Every call to run_async() gets a unique invocation_id (like "abc123")

When a tool pauses, you save this ID
When resuming, pass the same ID so ADK knows which execution to continue
Without it, ADK would start a NEW execution instead of resuming the paused one

Helper Functions to Process Events

These handle the event iteration logic for you.

check_for_approval() - Detects if the agent paused

Loops through all events and looks for the special adk_request_confirmation event
Returns approval_id (identifies this specific request) and invocation_id (identifies which execution to resume)
Returns None if no pause detected

## 2.4 : Helper Functions to Process Events
These handle the event iteration logic for us.

check_for_approval() - Detects if the agent paused

Loops through all events and looks for the special adk_request_confirmation event
Returns approval_id (identifies this specific request) and invocation_id (identifies which execution to resume)
Returns None if no pause detected

In [20]:
def check_for_approval(events):
    """Check if events contain an approval request.

    Returns:
        dict with approval details or None
    """
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if (
                    part.function_call
                    and part.function_call.name == "adk_request_confirmation"
                ):
                    return {
                        "approval_id": part.function_call.id,
                        "invocation_id": event.invocation_id,
                    }
    return None

print_agent_response() - Displays agent text

* Simple helper to extract and print text from events

In [21]:
def print_agent_response(events):
    """Print agent's text responses from events."""
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent > {part.text}")

create_approval_response() - Formats the human decision


* Takes the approval info and boolean decision (True/False) from the human
* Creates a FunctionResponse that ADK understands
* Wraps it in a Content object to send back to the agent


In [22]:
def create_approval_response(approval_info, approved):
    """Create approval response message."""
    confirmation_response = types.FunctionResponse(
        id=approval_info["approval_id"],
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(
        role="user", parts=[types.Part(function_response=confirmation_response)]
    )


print("✅ Helper functions defined")

✅ Helper functions defined


## 2.6 :The Workflow Function - Let's tie it all together!
The run_prescription_workflow() function orchestrates the entire approval flow.

Look for the code explanation in the cell below.

In [23]:
async def run_prescription_workflow(query: str, auto_approve: bool = True):
    """Runs  prescription workflow  with approval handling.

    Args:
        query: User's prescription request
        auto_approve: Whether to auto-approve large orders (simulates human decision)
    """

    print(f"\n{'='*60}")
    print(f"User > {query}\n")

    # Generate unique session ID
    session_id = f"order_{uuid.uuid4().hex[:8]}"

    # Create session
    await session_service.create_session(
        app_name="prescription_coordinator", user_id="test_user", session_id=session_id
    )

    query_content = types.Content(role="user", parts=[types.Part(text=query)])
    events = []

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 1: Send initial request to the Agent. If num_containers > 5, the Agent returns the special `adk_request_confirmation` event
    async for event in prescription_runner.run_async(
        user_id="test_user", session_id=session_id, new_message=query_content
    ):
        events.append(event)

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 2: Loop through all the events generated and check if `adk_request_confirmation` is present.
    approval_info = check_for_approval(events)

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    # STEP 3: If the event is present, it's a large order - HANDLE APPROVAL WORKFLOW
    if approval_info:
        print(f"⏸️  Pausing for approval...")
        print(f"🤔 Human Decision: {'APPROVE ✅' if auto_approve else 'REJECT ❌'}\n")

        # PATH A: Resume the agent by calling run_async() again with the approval decision
        async for event in prescription_runner.run_async(
            user_id="test_user",
            session_id=session_id,
            new_message=create_approval_response(
                approval_info, auto_approve
            ),  # Send human decision here
            invocation_id=approval_info[
                "invocation_id"
            ],  # Critical: same invocation_id tells ADK to RESUME
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent > {part.text}")

    # -----------------------------------------------------------------------------------------------
    # -----------------------------------------------------------------------------------------------
    else:
        # PATH B: If the `adk_request_confirmation` is not present - no approval needed - order completed immediately.
        print_agent_response(events)

    print(f"{'='*60}\n")


print("✅ Workflow function ready")

✅ Workflow function ready


Code breakdown
**Step 1: Send initial request to the Agent**

* Call run_async() to start agent execution
* Collect all events in a list for inspection
* 
**Step 2: Detect Pause**

* Call check_for_approval(events) to look for the special event: adk_request_confirmation
* Returns approval info (with invocation_id) if the special event is present; None if completed

**Step 3: Resume execution**

PATH A:

* If the approval info is present, at this point the Agent pauses for human input.
* Once the Human input is available, call the agent again using run_async() and pass in the Human input.
* Critical: Same invocation_id (tells ADK to RESUME, not restart)
* Display agent's final response after resuming

**PATH B:**

* If the approval info is not present, then approval is not needed and the agent completes execution.

In [24]:
from google.adk.runners import InMemoryRunner

# Initialize the runner and keep it in memory
#prescription_runner = InMemoryRunner(agent=root_agent)
#print("✅ Runner is initialized and ready.")

## 2.7:  Demo: Testing the Workflow
Now, let's run our demos. Notice how much cleaner and easier to read they are. All the complex logic for pausing and resuming is now hidden away in our run_workflow helper function, allowing us to focus on the tasks we want the agent to perform.

Note: You may see warnings like Warning: there are non-text parts in the response: ['function_call'] - this is normal and can be ignored. It just means the agent is calling tools in addition to generating text.

In [25]:
# Demo : Workflow simulates human decision: APPROVE ✅
await run_prescription_workflow("Prepare prescription 1 , approved", auto_approve=True)


User > Prepare prescription 1 , approved



/usr/local/lib/python3.11/dist-packages/google/adk/tools/tool_context.py:92: UserWarning: [EXPERIMENTAL] ToolConfirmation: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  ToolConfirmation(
/usr/local/lib/python3.11/dist-packages/google/adk/agents/invocation_context.py:298: UserWarning: [EXPERIMENTAL] BaseAgentState: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self.agent_states[event.author] = BaseAgentState()


⏸️  Pausing for approval...
🤔 Human Decision: APPROVE ✅




##  2.3 Run your agent
Now it's time to bring your agent to life and send it a query. To do this, you need a Runner, which is the central component within ADK that acts as the orchestrator. It manages the conversation, sends our messages to the agent, and handles its responses.

**a. Create an InMemoryRunner and tell it to use our root_agent:**

In [26]:
runner = InMemoryRunner(agent=froot_agent)

print("✅ Runner created.")

✅ Runner created.


b. Now you can call the .run_debug() method to send our prompt and get an answer.

👉 This method abstracts the process of session creation and maintenance and is used in prototyping.

ADD HILP

In [27]:
response = await runner.run_debug("Medicines for Throat inflamation")


 ### Created new session: debug_session_id

User > Medicines for Throat inflamation
AllopathicAgent > For throat inflammation, several allopathic medications can provide relief. The choice of medication often depends on the underlying cause of the inflammation.

**Over-the-Counter Pain Relievers:**

*   **Ibuprofen (e.g., Advil, Motrin):** This is a non-steroidal anti-inflammatory drug (NSAID) that helps reduce both pain and inflammation. A typical adult dosage is 200-400 mg every 4-6 hours as needed, not exceeding 1200 mg per day unless directed by a doctor.
*   **Acetaminophen (e.g., Tylenol):** This medication is effective for pain relief and reducing fever but does not reduce inflammation. The typical adult dosage is 325-1000 mg every 4-6 hours, not exceeding 4000 mg per day.

**Prescription Medications (for bacterial infections):**

*   **Amoxicillin:** If the sore throat is caused by a bacterial infection, such as strep throat, antibiotics like amoxicillin are prescribed. A comm

CriticAgent > APPROVED


In [28]:
from IPython.display import HTML, Markdown, display

# The response variable is a list that contains a mix of strings and Event objects.
# We must convert every item to a string before joining.
string_response_list = [str(item) for item in response]

# Now, join the list of strings.
full_response_text = "\n".join(string_response_list)

# Display the full string as Markdown.
display(Markdown(full_response_text))

model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""For throat inflammation, several allopathic medications can provide relief. The choice of medication often depends on the underlying cause of the inflammation.

**Over-the-Counter Pain Relievers:**

*   **Ibuprofen (e.g., Advil, Motrin):** This is a non-steroidal anti-inflammatory drug (NSAID) that helps reduce both pain and inflammation. A typical adult dosage is 200-400 mg every 4-6 hours as needed, not exceeding 1200 mg per day unless directed by a doctor.
*   **Acetaminophen (e.g., Tylenol):** This medication is effective for pain relief and reducing fever but does not reduce inflammation. The typical adult dosage is 325-1000 mg every 4-6 hours, not exceeding 4000 mg per day.

**Prescription Medications (for bacterial infections):**

*   **Amoxicillin:** If the sore throat is caused by a bacterial infection, such as strep throat, antibiotics like amoxicillin are prescribed. A common dosage for adults is 500 mg three times a day for 7-10 days, or 875 mg twice a day for 5-7 days, depending on the severity and specific infection.
*   **Penicillin:** Another first-line antibiotic for bacterial throat infections. A typical adult dosage might be 250 mg four times a day for 10 days.

**Other Symptomatic Relief:**

*   **Medicated Lozenges/Sprays:** Products containing ingredients like menthol, benzocaine, or phenol can help numb the throat and provide temporary pain relief. The dosage and frequency of use would be as directed on the product packaging.

It is important to consult a healthcare professional for a proper diagnosis and treatment plan, especially if symptoms are severe or persistent. Antibiotics are only effective against bacterial infections and will not help viral infections, which are the most common cause of sore throats."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        title='medicalnewstoday.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEwJJNA0QuLycRiiC2RCW_hFW4EK7biYQXiiO4jnjIYSnp85mZt-GMd9GkJO78KIzElGw14VlA75K1SehBwBFU1BZmJeBiLJJ-w7s_l-LbJQKxfQKUwH6QA2dmkBiwHd7ylcFT3TIE1rYwt-QqZeUPvUh_l_VVGDU45nNdxOroX-ZdVLyA='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='drugs.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGo3xx9L_L83k5K3UaalqPiITrzaDXtswEnlCyIIiKPzC6rgGbRxiV5zBjB79_LccXCCYYPW_Iui3ao0lQxKdETwR4EjU-9wQPq37VSPIlyvZzC2QVWHgDcQGUirA0zRP77jUbW615kthhSn81XiJfTtgSt4dd-0t7KAiN0BZl-agNi7SftbVyLdQ=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='clevelandclinic.org',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFaJPxmqbro7h47HE6uov5q_AxXgyOr-LloNoedEuf1YOxXG_7W_ZgppHRfbifxEj2BuTRKOap4yQ5HseAFF0vBcfSZW4tYIgrROAdhRqYdbTyEQfMbNoTgn1gXYFm3hf0EFJqXNDwLjl4pPmgK2V3E-Rf6Aq3rMYeJD697Rlk_8G6RtkLn'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='healthline.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGCkyJ5SKkyRQ9OLeKubPDkd0DDtm0Ya74CGzHdSLPyT0eUPGt_mW86J9rQL-qHQTsfUYylaMQZGTNuYFb1z5qEF6rWeTEqN66pk2gIj3-Biu6EpGjxReEahfL_Uj07U928FGfPCMV3bw=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='mayoclinic.org',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGFL2uWFz-Mqbw_jhlTb2GwXiJR0ez9Ry3tfuup36OWDYhzxxhQYUSyOnaUPN1AxA83TYk0aTwYD2ND87fnNdv6KXtjANX6lSDS4PZdg14WpHSlWdyvRkqUwVlGX9-8j9qru1BeDlATEd5TKMjS7cu0SuOsp8ThccAqb46nxnSvHCT936YrsIB1lPdk1L53S7ScmGyT94A='
      )
    ),
    <... 4 more items ...>,
  ],
  grounding_supports=[
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
        2,
      ],
      segment=Segment(
        end_index=463,
        start_index=341,
        text='A typical adult dosage is 200-400 mg every 4-6 hours as needed, not exceeding 1200 mg per day unless directed by a doctor.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        3,
        1,
        2,
        4,
      ],
      segment=Segment(
        end_index=688,
        start_index=601,
        text='The typical adult dosage is 325-1000 mg every 4-6 hours, not exceeding 4000 mg per day.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        3,
        1,
      ],
      segment=Segment(
        end_index=1046,
        start_index=891,
        text='A common dosage for adults is 500 mg three times a day for 7-10 days, or 875 mg twice a day for 5-7 days, depending on the severity and specific infection.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        3,
        1,
      ],
      segment=Segment(
        end_index=1198,
        start_index=1130,
        text='A typical adult dosage might be 250 mg four times a day for 10 days.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        5,
        6,
        7,
      ],
      segment=Segment(
        end_index=1393,
        start_index=1231,
        text='*   **Medicated Lozenges/Sprays:** Products containing ingredients like menthol, benzocaine, or phenol can help numb the throat and provide temporary pain relief.'
      )
    ),
    <... 1 more items ...>,
  ],
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG6hAeKXEzSECZiwPXzKDkNr4RZOEuaH__1PjfrljM8SjMZqf2qkdwW49US_fMWM09fSGaxc0X8agz2kz-KevF716J6qU-IMWienrHAg6sY3WmtqyFcsW4CBYguB7x6ViDfkx5zOX9xuZXcJlg_z9x9ZaP1QzA4rTiDs4EntXeONQYG30GwdCfW_5wPFJonEXPJUyKVN0BshXXNfIYhAQ==">medications for sore throat</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE2C9G7fHLINWqWQbkadaXQ_emgUzwklowYELiwNOVWL3wGSUeR-utXgCEI00GXv8wa4clKOoh1CimqPv6VXLcdHUG5Ntzme-NcFS_vPGxZdky7cTgztn3X3Px8zQvbjSrr-9M9OVALUCUgBR-WVO2iUq3RQOLytRR0mV5Iw_7t7CYYDJU3DOn926qb-0XrEczkCF42e9x-D2yqYwKdKnIv3xBfhlIOIBcxSX60TQI=">allopathic medicine for throat inflammation</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQ9NNLdVOt4JEAK3RWRpGJ11I-AFFPl34P1I7yh8XrQAxvZH7MjudxI3UwZCxfPJnNo0aRRRG17Sm15Zvghpng1mnHP0TmipYK4RmeuAf4eebPXlbFS89cKY72P6he6L5lMtm7iKM9z6sYD7JdIiEYm5pFyqA6ilGLfvhZURj7YdXgN-JYToNHB_0CIydpJrsKPKYr4y2nMLUGl7dhTalBiAJ_qswyebi8KZYEa-9kGGZ8114=">over-the-counter remedies for throat inflammation</a>
  </div>
</div>
"""
  ),
  web_search_queries=[
    'allopathic medicine for throat inflammation',
    'medications for sore throat',
    'over-the-counter remedies for throat inflammation',
  ]
) partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=453,
  prompt_token_count=68,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=68
    ),
  ],
  tool_use_prompt_token_count=103,
  tool_use_prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=103
    ),
  ],
  total_token_count=624
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='AllopathicAgent' actions=EventActions(skip_summarization=None, state_delta={'allopathic_findings': 'For throat inflammation, several allopathic medications can provide relief. The choice of medication often depends on the underlying cause of the inflammation.\n\n**Over-the-Counter Pain Relievers:**\n\n*   **Ibuprofen (e.g., Advil, Motrin):** This is a non-steroidal anti-inflammatory drug (NSAID) that helps reduce both pain and inflammation. A typical adult dosage is 200-400 mg every 4-6 hours as needed, not exceeding 1200 mg per day unless directed by a doctor.\n*   **Acetaminophen (e.g., Tylenol):** This medication is effective for pain relief and reducing fever but does not reduce inflammation. The typical adult dosage is 325-1000 mg every 4-6 hours, not exceeding 4000 mg per day.\n\n**Prescription Medications (for bacterial infections):**\n\n*   **Amoxicillin:** If the sore throat is caused by a bacterial infection, such as strep throat, antibiotics like amoxicillin are prescribed. A common dosage for adults is 500 mg three times a day for 7-10 days, or 875 mg twice a day for 5-7 days, depending on the severity and specific infection.\n*   **Penicillin:** Another first-line antibiotic for bacterial throat infections. A typical adult dosage might be 250 mg four times a day for 10 days.\n\n**Other Symptomatic Relief:**\n\n*   **Medicated Lozenges/Sprays:** Products containing ingredients like menthol, benzocaine, or phenol can help numb the throat and provide temporary pain relief. The dosage and frequency of use would be as directed on the product packaging.\n\nIt is important to consult a healthcare professional for a proper diagnosis and treatment plan, especially if symptoms are severe or persistent. Antibiotics are only effective against bacterial infections and will not help viral infections, which are the most common cause of sore throats.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch='ParallelResearchTeam.AllopathicAgent' id='f1ec73e8-016d-4f0a-a56c-462eb0066f42' timestamp=1763227751.696515
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""For throat inflammation, several homeopathic medicines can be considered, with dosages often depending on the severity of symptoms. It is always recommended to consult with a qualified homeopath for personalized advice. However, here are some commonly used remedies and their typical dosages:

*   **Belladonna**: This remedy is often indicated for sudden, intense sore throats with redness and swelling. It can be taken in 30C potency, 2-3 times a day, or even every 2 hours in acute cases with high fever.
*   **Hepar Sulph**: This is useful when there is a sensation of a splinter or fishbone in the throat, with extreme sensitivity to cold. A common dosage is 30C, taken 2-3 times daily.
*   **Mercurius Solubilis (Merc Sol)**: Indicated for sore throats with excessive salivation, foul breath, and ulcerated tonsils. It is typically given in 30C potency, 2-3 times daily until symptoms improve.
*   **Phytolacca Decandra**: Recommended for intense sore throats with a feeling of a lump or splinter, and pain that may radiate to the ears. A dosage of 30C, 3 times a day during the acute phase, is often suggested.
*   **Lachesis**: This remedy is considered for left-sided throat pain and a feeling of constriction or choking. It can be taken in 30C potency, once or twice a day, adjusted as symptoms change.
*   **Arsenicum Album**: Used for sore throats accompanied by a burning sensation, dryness, and thirst, often worse at night.
*   **Aconite**: This medicine is useful in the early stages of acute infection, especially with sudden onset of symptoms.
*   **ThroatCalm Tablets**: This is a combination homeopathic medicine for sore throat and hoarseness. Adults and children 6 years and older can take 2 tablets every 15 minutes for the first hour at the onset of symptoms, then every 6 hours, decreasing frequency with improvement. For children 4 to under 6 years, dissolve 2 tablets in 1 tablespoon of water and follow the same dosage instructions.
*   **Sore Throat Homeopathic Combo**: This combination remedy is typically taken as 1-2 pillules under the tongue or dissolved in water every 30-60 minutes until symptoms improve. Do not administer within 15 minutes of food or drink.

**Important Note:** Homeopathic treatment is individualized. The selection of the most appropriate remedy and its dosage depend on the specific symptoms and overall health of the individual. It is crucial to consult with a qualified homeopath or healthcare professional for accurate diagnosis and treatment."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        title='homeocareclinic.in',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEjLLsGei9bOKD-_vFyS9EXma4jYaMAUxedIjk-aOvPQqAjT1qdH_IoDGb9nZDOcUdR6eoR1x6HHN2Q7M8MucQgoqc4wKO0j5jU8NZh6_MeJtYdCdPM3bAA0wdk_OQLS6CZEP6hcE2y1ZmshX7Az38RvBXDibQ_vM_OkLCuaovhqmpfOV4fc1QvKmIn3H9JG7ieDqfdYg=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='drhomeo.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDYzdY1mhCwTaJpfntIMThTDZgzlyzglXdw9dQ8WQPAo_eNMQcgs_VAIagY0bHEdCaSEcs988zTC08vOVeJYmhHTC68CKJHF0ZF0TS9g4uTSrVxdl_qNoiTHZWfy03lG3sjVyHgWlwEVJ-yXFGSAyYKYavZ4Rdj9wG6KeYksl6oxEd6DIQjcClIh8='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='karenleadbeater.co.uk',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHPM14guTp8p7I5DmVLkC2IiNT0wdjwkIfiFmvwHiiNrsCBrRo0zqF_DVK4wcLjN2ernzPcdGsPKyT0nHRaSQmfZKVTtgWChTM84WmeQimKgScLLD6DiBZ0Y2w-W2O0JANXKA3EdTE-sppCzKSR_uHOIDkxMh6UncePGFGW6DYFaWmFLwQMPtSl_lHVh41Gv_KDMIs='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='homeopathyschool.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMGw37oijHmNAPYLeWvSLG4gbGTKiKfq7IcOPFXPwqJvCgW4Tux7dqpHMcFgV4zmcMx65zewUqBg0LY6FSGgIm1ab2qfcaVxdzN66UYwjfKpGZc3k0rwIa-VF9sVPU7EpBQgpfHNqzl0hT2wFd1rp6hrEdESNsiDupn0aA0JvKU-7c2a28PZhB'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='homeopathic.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHEN2oVRRPmOCZLQAt3d3D-jmhy4d41EHg_zMRqLvbkYs3-bJiiLzL2yFQW6y8AHvrWkfhSwwxmSABhMetH_ods235UWVBzN21Q0_xTKGlFAbzri4W2RqfGmpaMDqMskbQw2orBu8laQpVaJGJYt16dpBsg31fKOhQ='
      )
    ),
    <... 4 more items ...>,
  ],
  grounding_supports=[
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
      ],
      segment=Segment(
        end_index=507,
        start_index=405,
        text='It can be taken in 30C potency, 2-3 times a day, or even every 2 hours in acute cases with high fever.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        2,
      ],
      segment=Segment(
        end_index=691,
        start_index=645,
        text='A common dosage is 30C, taken 2-3 times daily.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        2,
      ],
      segment=Segment(
        end_index=899,
        start_index=822,
        text='It is typically given in 30C potency, 2-3 times daily until symptoms improve.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        3,
      ],
      segment=Segment(
        end_index=1117,
        start_index=1043,
        text='A dosage of 30C, 3 times a day during the acute phase, is often suggested.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        2,
      ],
      segment=Segment(
        end_index=1312,
        start_index=1231,
        text='It can be taken in 30C potency, once or twice a day, adjusted as symptoms change.'
      )
    ),
    <... 5 more items ...>,
  ],
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWPsuEPyalwp1xXf-MXwUB6CTkQigOgVlJ4at9AVepYONlMWGYMVOjEUQMh-WlX97MNgukTsMjdrZnVqgVe4ukSo40aSRPGV26yeM77KyS0UeXEkQIA-gyP1ipXuRK0iV4zxDJFVVuUmnp_-fc84JZr6x_Iv2SXza3wInB07TY2wTy1pC0FnhdN3C3fG1YWMND-CMnD9C29ON4QRB2eqQ-ZoVMNMVHXUQ_0bve8ds=">homeopathic treatment for pharyngitis dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEvM7pFfWkOV-xu5AfF-YPFzIe1rvAE2BqQTHnd7cJSybL6JvOS6Nw6dnvdX-UEGJb16X9_8lmo-vG2zzcKZaiQIBiWD1bT8xmBRSp2geEfPBQ5bq8u5yhKz6u6Z00z719Te7QNhlvp5Q-CslKJLKJFNki3L6NLHWpikv3CS37by7Um9cPUCRlJBlOYQWppwnQlI_riTz6MKQPD0pdYVVrF9ZTcfxRVcKuWf_aBMmniWG2_xkA8iA==">homeopathic medicine for throat inflammation dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFl7kl0khCtG8MfSs60HDYSVoSouVqd1lm5KfKg6XNVbyRZVWKlv7f0maaBU9XEQvebn38D2AIdcwN5CEhrFIOpJFpymSIGWk4dkzCWMPiLagUSeSaQQPY9VvmrOi8y8AiAQw4CmC64zXqEXZiFxyYfMXrGzhErYCt_xXOf6x-y_QxA7ltQ6giqXcgmTzBekspUFwjUhi0iP1DJvmbSh6A_ozjZyudCdYgUXyZ1AmQ-Ie8=">homeopathic remedies for sore throat and dosage</a>
  </div>
</div>
"""
  ),
  web_search_queries=[
    'homeopathic medicine for throat inflammation dosage',
    'homeopathic remedies for sore throat and dosage',
    'homeopathic treatment for pharyngitis dosage',
  ]
) partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=605,
  prompt_token_count=68,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=68
    ),
  ],
  tool_use_prompt_token_count=106,
  tool_use_prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=106
    ),
  ],
  total_token_count=779
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='HomeopathicAgent' actions=EventActions(skip_summarization=None, state_delta={'homeopathic_findings': 'For throat inflammation, several homeopathic medicines can be considered, with dosages often depending on the severity of symptoms. It is always recommended to consult with a qualified homeopath for personalized advice. However, here are some commonly used remedies and their typical dosages:\n\n*   **Belladonna**: This remedy is often indicated for sudden, intense sore throats with redness and swelling. It can be taken in 30C potency, 2-3 times a day, or even every 2 hours in acute cases with high fever.\n*   **Hepar Sulph**: This is useful when there is a sensation of a splinter or fishbone in the throat, with extreme sensitivity to cold. A common dosage is 30C, taken 2-3 times daily.\n*   **Mercurius Solubilis (Merc Sol)**: Indicated for sore throats with excessive salivation, foul breath, and ulcerated tonsils. It is typically given in 30C potency, 2-3 times daily until symptoms improve.\n*   **Phytolacca Decandra**: Recommended for intense sore throats with a feeling of a lump or splinter, and pain that may radiate to the ears. A dosage of 30C, 3 times a day during the acute phase, is often suggested.\n*   **Lachesis**: This remedy is considered for left-sided throat pain and a feeling of constriction or choking. It can be taken in 30C potency, once or twice a day, adjusted as symptoms change.\n*   **Arsenicum Album**: Used for sore throats accompanied by a burning sensation, dryness, and thirst, often worse at night.\n*   **Aconite**: This medicine is useful in the early stages of acute infection, especially with sudden onset of symptoms.\n*   **ThroatCalm Tablets**: This is a combination homeopathic medicine for sore throat and hoarseness. Adults and children 6 years and older can take 2 tablets every 15 minutes for the first hour at the onset of symptoms, then every 6 hours, decreasing frequency with improvement. For children 4 to under 6 years, dissolve 2 tablets in 1 tablespoon of water and follow the same dosage instructions.\n*   **Sore Throat Homeopathic Combo**: This combination remedy is typically taken as 1-2 pillules under the tongue or dissolved in water every 30-60 minutes until symptoms improve. Do not administer within 15 minutes of food or drink.\n\n**Important Note:** Homeopathic treatment is individualized. The selection of the most appropriate remedy and its dosage depend on the specific symptoms and overall health of the individual. It is crucial to consult with a qualified homeopath or healthcare professional for accurate diagnosis and treatment.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch='ParallelResearchTeam.HomeopathicAgent' id='12a6bb7a-5123-450c-a562-aaf602cb5bad' timestamp=1763227751.464529
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""Here are some Ayurvedic medicines and remedies for throat inflammation, along with their general dosages:

**Ayurvedic Medicines:**

*   **Vyoshadi Vatakam (Granules):** This is a traditional Ayurvedic remedy for sore throat, cough, and throat irritation. It helps ease bronchitis and other respiratory infections, supporting overall throat health. The dosage is typically 1-2 grams with honey or warm water after meals, or as directed by a physician.
*   **Sudarsana Tablets:** These tablets are ideal for relieving sore throat, fever, and chills. They possess antipyretic and analgesic qualities, helping to treat colds, coughs, and related respiratory discomfort. The dosage is usually 1-2 tablets, 3-4 times a day, or as directed by a physician.
*   **Talisadi Churna:** This formulation alleviates throat irritation by reducing Kapha and clearing mucus, making it highly effective for chronic throat infections. The recommended dosage is 1-2 grams with honey or warm water after meals.
*   **Sitopaladi Churna:** This powder balances Kapha and Vata doshas, providing relief from sore throat, dry cough, and congestion by acting as an expectorant. The dosage is typically 1-2 grams with honey, 2-3 times a day.
*   **Khadiradi Vati:** Known for its antibacterial and anti-inflammatory properties, this remedy soothes irritation, reduces swelling, and prevents infection in the throat. The dosage is 1-2 tablets to be sucked slowly, 3-4 times a day.
*   **Agasthya Rasayanam (Leyham):** This is a powerful remedy for throat pain, especially beneficial for chest congestion and chronic cough. It helps relieve respiratory discomfort and mucus buildup. The dosage is typically 10-15 ml twice a day or as advised by a practitioner.
*   **Koflet or Dabur Honitus (Syrups):** These Ayurvedic syrups combine herbs like Tulsi and Mulethi for throat pain relief and are specifically designed for respiratory and throat issues. Dosage is usually 1-2 tablets or teaspoons twice a day, or as directed by a physician.
*   **Septilin Tablets:** These tablets help with throat infections by boosting immunity and reducing inflammation. Dosage should be as directed by a physician.

**Home Remedies and Their Usage:**

*   **Turmeric and Salt Gargle:** Mix ½ teaspoon of turmeric powder and a pinch of salt in a glass of warm water. Gargle 2-3 times a day. This combination helps reduce swelling, fight inflammation, and clear out germs.
*   **Licorice (Yashtimadhu/Mulethi) Decoction or Syrup:** Licorice root has demulcent and anti-inflammatory properties that soothe the throat. You can boil licorice root in water to make a decoction for gargling or sipping, or take 1 teaspoon of licorice powder mixed with warm water or honey twice daily. A syrup can be made by mixing licorice powder with honey.
*   **Tulsi Tea:** Boil a few Tulsi leaves in water for 5-10 minutes. Strain and drink the tea 2-3 times a day, optionally adding honey or lemon. Tulsi has antibacterial, antiviral, and anti-inflammatory properties.
*   **Ginger Tea:** Boil fresh ginger slices in water. Drink the ginger tea, optionally adding honey. Ginger has anti-inflammatory and immune-boosting properties.
*   **Amla Juice:** Consume 15-20 ml of Amla juice with a teaspoon of honey twice a day to treat sore throat and cough.
*   **Lemon and Honey Mixture:** Mix a spoonful of lemon juice and honey in warm water and drink. This mixture soothes the throat and has antimicrobial effects.
*   **Triphala Gargle:** Boil 1 teaspoon of Triphala powder in water, let it cool, and gargle twice daily. Triphala helps reduce inflammation and soothes throat discomfort.

It is advisable to consult with an Ayurvedic practitioner for personalized advice and appropriate dosages based on your specific condition."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata(
  grounding_chunks=[
    GroundingChunk(
      web=GroundingChunkWeb(
        title='aatreyaayurved.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGSpJyIxDFN7rEMWd0NYpfNtJml5-VLE-9bQA26_x5lUuAVBgJoJYbppe_wx3hojP_el40H8EBAQeaGNGmOFDHJiIYCCiTodwNfP-4gDHJ-q-pCVjLJkKrDMW5caF5WOdnbWxSBbg_ZSD6TpEczn-ws72n_HJeQ-iIrZddRwLEb-cd3Pt3TUqng2fUm_bnb'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='rajputayurved.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGjq2tXNoiJvA0I2dzByNPmZ8q9zyRwAVMdeAaHyzQNpHcMbp9IqKJMzkuvXHYvxPbVPjZMwhH0VvpkD7dyJDLYE2EsKkDcbHYTtPrUFYCpU8vg99o3xBaCWOrfnT6OnZQmgpq94l_fp89Ztwr-SdKDlSF9bnAW7AJvNnCt'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='keralaayurveda.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHpyTQ7eOBp3J5Hpkvn_AT3BcCGXwIzNhF2wTBnHKVQExWXemfrT1XZTlPT3WNDI9KwAQSyHiLgLgen7uweP0BMsnxanE-5TFYFUEnX3NAftbBHFoM459On8EwlsZxFIOMAZ9B3aWZoZuxOFJoNyyHIjdIBXemJWtzSNwTNWRTBO5Q7qB1B_AoV8-uapg=='
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='mahaherbals.biz',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFciMk8zvyCV-9DzMdnt9atGGnZFuC-2QXmZvEWhV93-PidxdtSjhMXg9uoFrFBFx4Ra5JMte8f-aGWSFIhPwr_KcTtsW46fF5GSDCLCNn3EuHZkumlDeSX3zma_ze4DjDvJ8BuYE42S_bsHmL3DhL-iJ5Yc6_6O91AwMBXkWH9Xkp8xQBRNzxEs8zdzrj4uE4BMV3DeeABQEwwB_01CEPjOskPhfxFe8CZI_P1kqtn'
      )
    ),
    GroundingChunk(
      web=GroundingChunkWeb(
        title='ask-ayurveda.com',
        uri='https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH9aGCnOoteaFcjIkDDI_F97PxphRDeZR4_AViMjcLz52nPx1Bjr9qaYJ6pKXFCOeWhdTIXrtREx_Q6h52Q-jApaAfQhQe0V4iDY8exN4f3yKSfTcdaaO5JcXeR_6yUBCbujkb2cpe29Gc25KUBdQ0_D9JPwTKbtZ8EFb-I3MKXvXpgBlQz5AbIwc9S1iBLTitfQbVo_MQcLoVYvgY-rspXVeaXPbF1z1A='
      )
    ),
    <... 3 more items ...>,
  ],
  grounding_supports=[
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
      ],
      segment=Segment(
        end_index=451,
        start_index=349,
        text='The dosage is typically 1-2 grams with honey or warm water after meals, or as directed by a physician.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        2,
        3,
      ],
      segment=Segment(
        end_index=749,
        start_index=667,
        text='The dosage is usually 1-2 tablets, 3-4 times a day, or as directed by a physician.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
      ],
      segment=Segment(
        end_index=990,
        start_index=917,
        text='The recommended dosage is 1-2 grams with honey or warm water after meals.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
        1,
      ],
      segment=Segment(
        end_index=1214,
        start_index=1152,
        text='The dosage is typically 1-2 grams with honey, 2-3 times a day.'
      )
    ),
    GroundingSupport(
      grounding_chunk_indices=[
        0,
      ],
      segment=Segment(
        end_index=1452,
        start_index=1389,
        text='The dosage is 1-2 tablets to be sucked slowly, 3-4 times a day.'
      )
    ),
    <... 10 more items ...>,
  ],
  search_entry_point=SearchEntryPoint(
    rendered_content="""<style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFs-HlG8B4NuQdpjKvnZIrOQHh-wpKA3hXMhCp7bVQ_d4IpVbuHr8CpE59dG-2hTJjRiOs1_TOO5l1qAsA9onpEGEOQslDXv-GHULDr41AqbeM_dEvKjciDrfVCNqA5zofBIzZwgR2lpgFONBtuxaP6-NZc0ddc-YRmuHujkwIkJDfUlA83JsbuFO6IvQYTGi_-y0wdr5noD0siubaTlEuCeClTUSMqtVghZ9ujDp7mXji7PA59-7tHd6I=">Ayurvedic medicine for throat inflammation with dosage</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFFzK7TZnkGKoNYrkVJnYM9pkTsBnQ8NbOGe3jiHTzoNnOG08CPMWT5t_xs6aqOU0nzQ13ylZ-GbdnW7J92T5P6J2LZAhMD6GzS_6fiQBaBlLJd-qDCKfwJJox8ldO7hTrbP4LZB5tNOETzPL0rnppbW0APWx074KD2kSc_j4gH3qameq4jHOpUwpRv84hRek3uzy-DkkNVcUE243ukcJy2rs1hosRZooRDQvJc5Yt8EVA=">Ayurvedic remedies for sore throat with dosage</a>
  </div>
</div>
"""
  ),
  web_search_queries=[
    'Ayurvedic medicine for throat inflammation with dosage',
    'Ayurvedic remedies for sore throat with dosage',
  ]
) partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=890,
  prompt_token_count=67,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=67
    ),
  ],
  tool_use_prompt_token_count=98,
  tool_use_prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=98
    ),
  ],
  total_token_count=1055
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='AyurvedicAgent' actions=EventActions(skip_summarization=None, state_delta={'ayurvedic_findings': 'Here are some Ayurvedic medicines and remedies for throat inflammation, along with their general dosages:\n\n**Ayurvedic Medicines:**\n\n*   **Vyoshadi Vatakam (Granules):** This is a traditional Ayurvedic remedy for sore throat, cough, and throat irritation. It helps ease bronchitis and other respiratory infections, supporting overall throat health. The dosage is typically 1-2 grams with honey or warm water after meals, or as directed by a physician.\n*   **Sudarsana Tablets:** These tablets are ideal for relieving sore throat, fever, and chills. They possess antipyretic and analgesic qualities, helping to treat colds, coughs, and related respiratory discomfort. The dosage is usually 1-2 tablets, 3-4 times a day, or as directed by a physician.\n*   **Talisadi Churna:** This formulation alleviates throat irritation by reducing Kapha and clearing mucus, making it highly effective for chronic throat infections. The recommended dosage is 1-2 grams with honey or warm water after meals.\n*   **Sitopaladi Churna:** This powder balances Kapha and Vata doshas, providing relief from sore throat, dry cough, and congestion by acting as an expectorant. The dosage is typically 1-2 grams with honey, 2-3 times a day.\n*   **Khadiradi Vati:** Known for its antibacterial and anti-inflammatory properties, this remedy soothes irritation, reduces swelling, and prevents infection in the throat. The dosage is 1-2 tablets to be sucked slowly, 3-4 times a day.\n*   **Agasthya Rasayanam (Leyham):** This is a powerful remedy for throat pain, especially beneficial for chest congestion and chronic cough. It helps relieve respiratory discomfort and mucus buildup. The dosage is typically 10-15 ml twice a day or as advised by a practitioner.\n*   **Koflet or Dabur Honitus (Syrups):** These Ayurvedic syrups combine herbs like Tulsi and Mulethi for throat pain relief and are specifically designed for respiratory and throat issues. Dosage is usually 1-2 tablets or teaspoons twice a day, or as directed by a physician.\n*   **Septilin Tablets:** These tablets help with throat infections by boosting immunity and reducing inflammation. Dosage should be as directed by a physician.\n\n**Home Remedies and Their Usage:**\n\n*   **Turmeric and Salt Gargle:** Mix ½ teaspoon of turmeric powder and a pinch of salt in a glass of warm water. Gargle 2-3 times a day. This combination helps reduce swelling, fight inflammation, and clear out germs.\n*   **Licorice (Yashtimadhu/Mulethi) Decoction or Syrup:** Licorice root has demulcent and anti-inflammatory properties that soothe the throat. You can boil licorice root in water to make a decoction for gargling or sipping, or take 1 teaspoon of licorice powder mixed with warm water or honey twice daily. A syrup can be made by mixing licorice powder with honey.\n*   **Tulsi Tea:** Boil a few Tulsi leaves in water for 5-10 minutes. Strain and drink the tea 2-3 times a day, optionally adding honey or lemon. Tulsi has antibacterial, antiviral, and anti-inflammatory properties.\n*   **Ginger Tea:** Boil fresh ginger slices in water. Drink the ginger tea, optionally adding honey. Ginger has anti-inflammatory and immune-boosting properties.\n*   **Amla Juice:** Consume 15-20 ml of Amla juice with a teaspoon of honey twice a day to treat sore throat and cough.\n*   **Lemon and Honey Mixture:** Mix a spoonful of lemon juice and honey in warm water and drink. This mixture soothes the throat and has antimicrobial effects.\n*   **Triphala Gargle:** Boil 1 teaspoon of Triphala powder in water, let it cool, and gargle twice daily. Triphala helps reduce inflammation and soothes throat discomfort.\n\nIt is advisable to consult with an Ayurvedic practitioner for personalized advice and appropriate dosages based on your specific condition.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch='ParallelResearchTeam.AyurvedicAgent' id='8d72c2e0-f972-481a-9b69-dad657e1d82b' timestamp=1763227751.218731
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""**Executive Prescription for Throat Inflammation:**

**Homeopathic:** Belladonna 30C (2-3x/day), Hepar Sulph 30C (2-3x/day), Merc Sol 30C (2-3x/day).
**Ayurvedic:** Vyoshadi Vatakam (1-2g w/ honey), Khadira Vati (1-2 tabs to suck), Turmeric/Salt Gargle.
**Allopathic:** Ibuprofen 200-400mg (q4-6h) or Acetaminophen 325-1000mg (q4-6h). Antibiotics (Amoxicillin/Penicillin) if bacterial.

This is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes."""
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=176,
  prompt_token_count=3846,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3846
    ),
  ],
  total_token_count=4022
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='AggregatorAgent' actions=EventActions(skip_summarization=None, state_delta={'prescription_summary': '**Executive Prescription for Throat Inflammation:**\n\n**Homeopathic:** Belladonna 30C (2-3x/day), Hepar Sulph 30C (2-3x/day), Merc Sol 30C (2-3x/day).\n**Ayurvedic:** Vyoshadi Vatakam (1-2g w/ honey), Khadira Vati (1-2 tabs to suck), Turmeric/Salt Gargle.\n**Allopathic:** Ibuprofen 200-400mg (q4-6h) or Acetaminophen 325-1000mg (q4-6h). Antibiotics (Amoxicillin/Penicillin) if bacterial.\n\nThis is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='890eba98-6de5-4cf9-9f44-1687ea825397' timestamp=1763227756.614218
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text="""The provided prescription is a combination of Homeopathic, Ayurvedic, and Allopathic treatments for throat inflammation. While it includes a range of options, it lacks crucial elements for a well-written and complete medical prescription.

Here are specific, actionable suggestions for improvement:

1.  **Lack of Specificity for Antibiotics:** The prescription mentions "Antibiotics (Amoxicillin/Penicillin) if bacterial" but fails to specify the dosage, frequency, and duration of these antibiotics. This is critical for effective treatment and preventing antibiotic resistance. For example, a typical adult dosage for Amoxicillin might be 500mg three times a day for 7-10 days, or 875mg twice a day for 5-7 days, but this needs to be explicitly stated and, more importantly, *only prescribed after a medical professional has diagnosed a bacterial infection*.

2.  **Absence of Citations:** A comprehensive medical prescription should ideally include citations for the recommended treatments, especially for over-the-counter and prescription medications. This helps validate the recommendations and provides a basis for the chosen dosages. For instance, for the allopathic medications, it would be beneficial to cite reputable medical guidelines or drug information resources.

3.  **Incomplete Dosage and Usage Details for Ayurvedic/Homeopathic Remedies:** While dosages are provided for some, others could be more detailed. For example, "Vyoshadi Vatakam (1-2g w/ honey)" is good, but it's unclear if this is before or after meals, or how many times a day. Similarly, for "Khadira Vati (1-2 tabs to suck)," specifying the frequency (e.g., "3-4 times a day") would be beneficial. The "Turmeric/Salt Gargle" would benefit from clearer instructions on water temperature and the ratio of ingredients."""
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=383,
  prompt_token_count=2373,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2373
    ),
  ],
  total_token_count=2756
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='CriticAgent' actions=EventActions(skip_summarization=None, state_delta={'critique': 'The provided prescription is a combination of Homeopathic, Ayurvedic, and Allopathic treatments for throat inflammation. While it includes a range of options, it lacks crucial elements for a well-written and complete medical prescription.\n\nHere are specific, actionable suggestions for improvement:\n\n1.  **Lack of Specificity for Antibiotics:** The prescription mentions "Antibiotics (Amoxicillin/Penicillin) if bacterial" but fails to specify the dosage, frequency, and duration of these antibiotics. This is critical for effective treatment and preventing antibiotic resistance. For example, a typical adult dosage for Amoxicillin might be 500mg three times a day for 7-10 days, or 875mg twice a day for 5-7 days, but this needs to be explicitly stated and, more importantly, *only prescribed after a medical professional has diagnosed a bacterial infection*.\n\n2.  **Absence of Citations:** A comprehensive medical prescription should ideally include citations for the recommended treatments, especially for over-the-counter and prescription medications. This helps validate the recommendations and provides a basis for the chosen dosages. For instance, for the allopathic medications, it would be beneficial to cite reputable medical guidelines or drug information resources.\n\n3.  **Incomplete Dosage and Usage Details for Ayurvedic/Homeopathic Remedies:** While dosages are provided for some, others could be more detailed. For example, "Vyoshadi Vatakam (1-2g w/ honey)" is good, but it\'s unclear if this is before or after meals, or how many times a day. Similarly, for "Khadira Vati (1-2 tabs to suck)," specifying the frequency (e.g., "3-4 times a day") would be beneficial. The "Turmeric/Salt Gargle" would benefit from clearer instructions on water temperature and the ratio of ingredients.'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='f4e8b164-89f2-4558-a51c-672960fbc261' timestamp=1763227758.166489
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={},
        id='adk-34c6ff47-3ca3-4e13-b309-56c05897d554',
        name='exit_loop'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=3171,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3171
    ),
  ],
  total_token_count=3181
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=set() branch=None id='26d59203-7b10-4f33-ade5-e5aa7ec672b9' timestamp=1763227760.589667
model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='adk-34c6ff47-3ca3-4e13-b309-56c05897d554',
        name='exit_loop',
        response={
          'message': 'Prescription approved. Exiting refinement loop.',
          'status': 'approved'
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='1ba2a0be-7de6-4a1c-95fe-4299b5318295' timestamp=1763227761.581437
model_version='gemini-2.5-flash-lite' content=Content(
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=3209,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3209
    ),
  ],
  total_token_count=3209
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='9c53eaba-6d57-4e75-904e-338ad03b435e' timestamp=1763227761.583036
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      text='APPROVED'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=1,
  prompt_token_count=2815,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2815
    ),
  ],
  total_token_count=2816
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='CriticAgent' actions=EventActions(skip_summarization=None, state_delta={'critique': 'APPROVED'}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='6a320ab9-90c5-45e8-aa8d-d9c71516e15c' timestamp=1763227761.948383
model_version='gemini-2.5-flash-lite' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={},
        id='adk-f446db7f-0eb8-43e6-b2e3-f7afcd11ac1a',
        name='exit_loop'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=2838,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2838
    ),
  ],
  total_token_count=2848
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=set() branch=None id='7df3f7ca-bf88-4f56-8e25-44ba307cd215' timestamp=1763227762.427176
model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='adk-f446db7f-0eb8-43e6-b2e3-f7afcd11ac1a',
        name='exit_loop',
        response={
          'message': 'Prescription approved. Exiting refinement loop.',
          'status': 'approved'
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='f3628244-1bfb-42d4-bf07-47e0c04006d2' timestamp=1763227763.40607
model_version='gemini-2.5-flash-lite' content=Content(
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  prompt_token_count=2876,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2876
    ),
  ],
  total_token_count=2876
) live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None invocation_id='e-af888521-92ab-40b3-b6a3-9889df275854' author='RefinerAgent' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None) long_running_tool_ids=None branch=None id='ad7f7d4b-700a-4d05-a05d-649b06d44265' timestamp=1763227763.408354